# Stage 2 — YOLOv8n Phone/Screen Detector
Detects mobile phones, tablet screens, and persons shown on screens.
Short-circuits Stage 3 if a device is detected near the face.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics","roboflow"],check=True)
print("Deps installed")

In [ ]:
import torch, os
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# Download Mobile Person Dataset via Roboflow
import os
API_KEY = os.getenv("ROBOFLOW_API_KEY", "")
if API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=API_KEY)
    rf.workspace("shubham-vishwakarma-5olb4").project("mobile-person-datset").version(1)\n      .download("yolov8", location="./data/mobile_person")
else:
    print("Set ROBOFLOW_API_KEY to auto-download. Manual: universe.roboflow.com")

In [ ]:
# Auto-generate data.yaml
import sys; sys.path.insert(0, ".")
from training.train_detector import generate_data_yaml
from training.config import DetectorConfig
cfg = DetectorConfig()
yaml_path = generate_data_yaml(cfg)
print(f"YAML: {yaml_path}")

In [ ]:
# Train YOLOv8n (80 epochs)
from training.train_detector import train
best_weights = train(cfg, yaml_path)
print(f"Best weights: {best_weights}")

In [ ]:
# Show training curves
import pandas as pd, matplotlib.pyplot as plt
results_csv = "runs/detector/train/results.csv"
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["    train/box_loss"], label="box_loss"); axes[0].set_title("Train Loss")
    axes[1].plot(df["       metrics/mAP50(B)"], label="mAP50"); axes[1].set_title("Val mAP50")
    [ax.legend() for ax in axes]; plt.tight_layout(); plt.show()

In [ ]:
# Validate mAP50
from training.train_detector import validate as yolo_validate
map50 = yolo_validate(cfg, best_weights)
print(f"mAP50: {map50:.4f} (target > 0.80)")

In [ ]:
# Export to ONNX
from training.train_detector import export_onnx
onnx = export_onnx(best_weights, output_dir="/kaggle/working")
print(f"ONNX: {onnx}")

In [ ]:
print("Stage 2 training complete.")
print("Copy best_screen_detector.pt and .onnx to checkpoints/ in your project.")